In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"/")
AG_ROOT = ROOT / "Finished sections" / "Agriculture"

In [3]:
ROOT = Path(r"/")
AG_ROOT = ROOT / "Finished sections" / "Agriculture"

key_hh = ["Wave", "HHID"]
key_season = ["Wave", "HHID", "SEASON"]
key_crop = ["Wave", "HHID", "SEASON", "CROP_ID"]

def load_ag(name):
    df = pd.read_csv(AG_ROOT / name, dtype=str)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"WAVE": "Wave", "VISIT": "SEASON"})
    df["Wave"] = pd.to_numeric(df["Wave"], errors="coerce").astype("Int64")
    df["HHID"] = df["HHID"].astype("string").str.strip()
    return df

def num(s):
    return pd.to_numeric(s, errors="coerce")

def clean_code(s):
    return (
        s.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

def yes_1(x):
    return str(x).strip() in ["1", "1.0"]

ag2 = load_ag("AGSEC2_standardized.csv")
ag2["PARCEL_ID"] = clean_code(ag2["PARCEL_ID"])

ag2["A2Q4_num"] = num(ag2["A2Q4"])

# q19: 1 if 1, 2, or missing; 0 if 3
a2q19 = ag2["A2Q19"].astype("string").str.strip()
ag2["A2Q19_dummy"] = 1
ag2.loc[a2q19.isin(["3", "3.0"]), "A2Q19_dummy"] = 0

# q20 rainfed: 1 if 2; 0 if 1, 3, or missing
a2q20 = ag2["A2Q20"].astype("string").str.strip()
ag2["A2Q20_rainfed"] = 0
ag2.loc[a2q20.isin(["2", "2.0"]), "A2Q20_rainfed"] = 1

ag2_parcel = (
    ag2
    .groupby(["Wave", "HHID", "PARCEL_ID"], as_index=False)
    .agg(
        A2Q4=("A2Q4_num", "first"),
        A2Q19_dummy=("A2Q19_dummy", "mean"),
        A2Q20_rainfed=("A2Q20_rainfed", "mean"),
    )
)

ag2_hh = (
    ag2_parcel
    .groupby(key_hh, as_index=False)
    .agg(
        total_area=("A2Q4", "sum"),
        A2Q19_average=("A2Q19_dummy", "mean"),
        A2Q20_average_rainfed=("A2Q20_rainfed", "mean"),
    )
)


ag3 = load_ag("AGSEC3_standardized.csv")

for c in ["A3AQ4", "A3AQ14", "A3AQ26", "A3AQ38", "A3AQ39", "A3AQ43"]:
    ag3[f"{c}_num"] = num(ag3[c])

for c in ["A3AQ4", "A3AQ14", "A3AQ26"]:
    ag3[f"{c}_dummy"] = ag3[f"{c}_num"].eq(1).astype("Int64")

ag3["A3AQ41_any"] = ag3["A3AQ41"].map(lambda x: 1 if yes_1(x) else 0)

ag3_hh = (
    ag3
    .groupby(key_season, as_index=False)
    .agg(
        A3Q4_average=("A3AQ4_dummy", "mean"),
        A3Q14_average=("A3AQ14_dummy", "mean"),
        A3Q26_average=("A3AQ26_dummy", "mean"),
        A3AQ38_average=("A3AQ38_num", "mean"),
        A3Q39_average=("A3AQ39_num", "mean"),
        A3Q41_any_labor=("A3AQ41_any", "max"),
        A3Q43_sum=("A3AQ43_num", "sum"),
    )
)

ag4 = load_ag("AGSEC4_standardized.csv")
ag4["CROP_ID"] = clean_code(ag4["CROP_ID"])
ag4["A4AQ8_num"] = num(ag4["A4AQ8"])
ag4["A4AQ9_num"] = num(ag4["A4AQ9"])

intercrop = ag4["A4AQ7"].astype("string").str.strip().isin(["2", "2.0"])

ag4["crop_area"] = ag4["A4AQ8_num"]
ag4.loc[intercrop & ag4["A4AQ9_num"].notna(), "crop_area"] = (
    ag4["A4AQ8_num"] * ag4["A4AQ9_num"] / 100
)

ag4_crop = (
    ag4[ag4["CROP_ID"].notna()]
    .groupby(key_crop, as_index=False)
    .agg(
        planted_yes=("CROP_ID", "size"),
        crop_area=("crop_area", "sum"),
    )
)

ag4_crop["planted_yes"] = 1

ag5 = load_ag("AGSEC5_standardized.csv")
ag5["CROP_ID"] = clean_code(ag5["CROP_ID"])

def clean_numeric_value(s, missing_codes=[99999, 99998, 99997, 99996]):
    x = pd.to_numeric(s, errors="coerce")
    return x.mask(x.isin(missing_codes))


def sum_with_missing(s):
    return s.sum(min_count=1)


def first_nonmissing(s):
    s = s.dropna()

    if len(s) == 0:
        return pd.NA

    return s.iloc[0]


ag5["qty_harvested_units"] = clean_numeric_value(ag5["A5AQ6A"])
ag5["harvest_kg"] = clean_numeric_value(ag5["A5AQ6D"])

# If reported harvest quantity is zero, kg harvest should also be zero.
ag5.loc[ag5["qty_harvested_units"].eq(0), "harvest_kg"] = 0

ag5["seed_saved_units"] = clean_numeric_value(ag5["A5AQ15"])
ag5["lost_wasted_units"] = clean_numeric_value(ag5["A5AQ16"])

ag5["no_harvest_reason_A5Q24"] = clean_numeric_value(ag5["A5AQ24"])

ag5_crop = (
    ag5[ag5["CROP_ID"].notna()]
    .groupby(key_crop, as_index=False)
    .agg(
        qty_harvested_units=("qty_harvested_units", sum_with_missing),
        harvest_kg=("harvest_kg", sum_with_missing),
        seed_saved_units=("seed_saved_units", sum_with_missing),
        lost_wasted_units=("lost_wasted_units", sum_with_missing),
        no_harvest_reason_A5Q24=("no_harvest_reason_A5Q24", first_nonmissing),
    )
)

#===================================

def clean_merge_code(s):
    return (
        s.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace(["", "nan", "NaN", "<NA>"], pd.NA)
    )


def normalize_crop_keys(df):
    df = df.copy()

    df["Wave"] = pd.to_numeric(df["Wave"], errors="coerce").astype("Int64")
    df["SEASON"] = pd.to_numeric(df["SEASON"], errors="coerce").astype("Int64")
    df["HHID"] = clean_merge_code(df["HHID"])
    df["CROP_ID"] = clean_merge_code(df["CROP_ID"])

    return df


ag4_crop = normalize_crop_keys(ag4_crop)
ag5_crop = normalize_crop_keys(ag5_crop)

key_crop = ["Wave", "HHID", "SEASON", "CROP_ID"]

ag4_crop = (
    ag4_crop
    .groupby(key_crop, as_index=False)
    .agg(
        planted_yes=("planted_yes", "max"),
        crop_area=("crop_area", "sum"),
    )
)

ag5_crop = (
    ag5_crop
    .groupby(key_crop, as_index=False)
    .agg(
        qty_harvested_units=("qty_harvested_units", sum_with_missing),
        harvest_kg=("harvest_kg", sum_with_missing),
        seed_saved_units=("seed_saved_units", sum_with_missing),
        lost_wasted_units=("lost_wasted_units", sum_with_missing),
        no_harvest_reason_A5Q24=("no_harvest_reason_A5Q24", first_nonmissing),
    )
)

match_check = ag4_crop[key_crop].merge(
    ag5_crop[key_crop],
    on=key_crop,
    how="outer",
    indicator=True
)

display(
    match_check["_merge"]
    .value_counts(dropna=False)
    .reset_index(name="rows")
)

#===================================

crop_panel_long = ag4_crop.merge(
    ag5_crop,
    on=key_crop,
    how="outer",
    validate="1:1"
)

crop_panel_long = crop_panel_long.merge(
    ag2_hh,
    on=key_hh,
    how="left",
    validate="m:1"
)

crop_panel_long["share_crop_area_of_total_area"] = (
    crop_panel_long["crop_area"] / crop_panel_long["total_area"]
)

crop_panel_long["total_planted_area_season"] = (
    crop_panel_long
    .groupby(key_season)["crop_area"]
    .transform("sum")
)

crop_panel_long["share_crop_area_of_planted_area"] = (
    crop_panel_long["crop_area"] / crop_panel_long["total_planted_area_season"]
)

crop_panel_long["total_harvest_kg_season"] = (
    crop_panel_long
    .groupby(key_season)["harvest_kg"]
    .transform("sum")
)

crop_panel_long["share_crop_harvest_kg"] = (
    crop_panel_long["harvest_kg"] / crop_panel_long["total_harvest_kg_season"]
)

display(crop_panel_long.head())

,_merge,rows
0,both,102642
1,left_only,9886
2,right_only,5392


,Wave,HHID,SEASON,CROP_ID,planted_yes,crop_area,qty_harvested_units,harvest_kg,seed_saved_units,lost_wasted_units,no_harvest_reason_A5Q24,total_area,A2Q19_average,A2Q20_average_rainfed,share_crop_area_of_total_area,total_planted_area_season,share_crop_area_of_planted_area,total_harvest_kg_season,share_crop_harvest_kg
0,1,1.021E+11,1,130,1.0,0.825,NaN,NaN,NaN,NaN,NaN,11.89,1.0,0.9,0.069386,5.25,0.157143,0.0,NaN
1,1,1.021E+11,1,210,1.0,0.450,NaN,NaN,NaN,NaN,NaN,11.89,1.0,0.9,0.037847,5.25,0.085714,0.0,NaN
2,1,1.021E+11,1,310,1.0,0.850,NaN,NaN,NaN,NaN,NaN,11.89,1.0,0.9,0.071489,5.25,0.161905,0.0,NaN
3,1,1.021E+11,1,620,1.0,0.250,NaN,NaN,NaN,NaN,NaN,11.89,1.0,0.9,0.021026,5.25,0.047619,0.0,NaN
4,1,1.021E+11,1,630,1.0,0.575,NaN,NaN,NaN,NaN,NaN,11.89,1.0,0.9,0.048360,5.25,0.109524,0.0,NaN


In [5]:
crop_freq_source = crop_panel_long.copy()

crop_freq_source["CROP_ID_clean"] = (
    crop_freq_source["CROP_ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

crop_freq_source = crop_freq_source[
    crop_freq_source["CROP_ID_clean"].notna()
    & ~crop_freq_source["CROP_ID_clean"].isin(["", "nan", "NaN", "<NA>"])
].copy()

crop_freq_source["Wave"] = pd.to_numeric(crop_freq_source["Wave"], errors="coerce").astype("Int64")

def crop_sort_key(x):
    x = str(x)
    return int(x) if x.isdigit() else 999999

# Total frequency
crop_freq_total = (
    crop_freq_source
    .groupby("CROP_ID_clean")
    .size()
    .reset_index(name="freq_total")
)

crop_freq_by_wave = (
    crop_freq_source
    .groupby(["CROP_ID_clean", "Wave"])
    .size()
    .reset_index(name="freq")
    .pivot(index="CROP_ID_clean", columns="Wave", values="freq")
    .fillna(0)
)

crop_freq_by_wave.columns = [
    f"freq_wave{int(w)}"
    for w in crop_freq_by_wave.columns
]

for wave in [1, 2, 3, 4, 5, 7, 8]:
    col = f"freq_wave{wave}"
    if col not in crop_freq_by_wave.columns:
        crop_freq_by_wave[col] = 0

crop_frequency_table = (
    crop_freq_total
    .merge(crop_freq_by_wave.reset_index(), on="CROP_ID_clean", how="left")
    .rename(columns={"CROP_ID_clean": "Crop"})
)

wave_cols = [f"freq_wave{w}" for w in [1, 2, 3, 4, 5, 7, 8]]

crop_frequency_table = crop_frequency_table[
    ["Crop", "freq_total"] + wave_cols
]

crop_frequency_table = (
    crop_frequency_table
    .sort_values("Crop", key=lambda s: s.map(crop_sort_key))
    .reset_index(drop=True)
)

for col in ["freq_total"] + wave_cols:
    crop_frequency_table[col] = crop_frequency_table[col].astype(int)

display(crop_frequency_table)

,Crop,freq_total,freq_wave1,freq_wave2,freq_wave3,freq_wave4,freq_wave5,freq_wave7,freq_wave8
0,0,1,0,0,1,0,0,0,0
1,2,13,13,0,0,0,0,0,0
2,3,1,1,0,0,0,0,0,0
3,4,2,2,0,0,0,0,0,0
4,6,5,5,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
84,940,88,22,66,0,0,0,0,0
85,950,8,6,2,0,0,0,0,0
86,960,124,83,41,0,0,0,0,0
87,990,38,8,30,0,0,0,0,0


In [4]:
# Crop panel long -> wide HHID-wave-visit crop panel

crop_keep = [
    "111", "112", "120", "130", "141", "150",
    "210", "221", "222", "223", "224",
    "310", "320", "330", "340",
    "410", "420", "430", "440", "450", "460", "470",
    "510", "520", "530",
    "610", "620", "630", "640", "650",
    "700", "710", "720", "741", "742", "744", "750", "760", "770", "780", "790",
    "810", "820", "830", "840", "860", "870", "890",
]

kcal_per_kg = {
    "111": 3200, "112": 3400, "120": 3600, "130": 3050, "141": 3780, "150": 3390,
    "210": 500, "221": 810, "222": 810, "223": 810, "224": 810,
    "310": 4300, "320": 1700, "330": 0, "340": 4800,
    "410": 250, "420": 180, "430": 413, "440": 400, "450": 260, "460": 1200, "470": 250,
    "510": 580, "520": 0, "530": 0,
    "610": 770, "620": 860, "630": 1600, "640": 1180, "650": 1200,
    "700": 470, "710": 800, "720": 500, "741": 900, "742": 425, "744": 850,
    "750": 600, "760": 950, "770": 1400, "780": 970, "790": 0,
    "810": 0, "820": 5400, "830": 0, "840": 630, "860": 0, "870": 2800, "890": 0,
}

def sum_with_missing(s):
    return s.sum(min_count=1)

def clean_crop_code(s):
    return (
        s.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace(["", "nan", "NaN", "<NA>"], pd.NA)
    )

crop = crop_panel_long.copy()

crop["Wave"] = pd.to_numeric(crop["Wave"], errors="coerce").astype("Int64")
crop["SEASON"] = pd.to_numeric(crop["SEASON"], errors="coerce").astype("Int64")
crop["HHID"] = crop["HHID"].astype("string").str.strip()
crop["CROP_ID"] = clean_crop_code(crop["CROP_ID"])

# Aggregate variants into coffee all / 810
crop["CROP_ID"] = crop["CROP_ID"].replace({
    "811": "810",
    "812": "810",
})

crop = crop[crop["CROP_ID"].isin(crop_keep)].copy()

for col in ["crop_area", "harvest_kg"]:
    crop[col] = pd.to_numeric(crop[col], errors="coerce")

key_crop = ["Wave", "HHID", "SEASON", "CROP_ID"]
key_visit = ["Wave", "HHID", "SEASON"]

# Re-aggregate after recoding 811/812 into 810
crop_agg = (
    crop
    .groupby(key_crop, as_index=False)
    .agg(
        dummy_planted=("CROP_ID", "size"),
        sum_crop_area=("crop_area", sum_with_missing),
        sum_harvest_kg=("harvest_kg", sum_with_missing),
    )
)

crop_agg["dummy_planted"] = 1

crop_agg["kcal_per_kg"] = crop_agg["CROP_ID"].map(kcal_per_kg)
crop_agg["calories"] = crop_agg["sum_harvest_kg"] * crop_agg["kcal_per_kg"]

crop_agg["total_planted_area_visit"] = (
    crop_agg
    .groupby(key_visit)["sum_crop_area"]
    .transform(lambda s: s.sum(min_count=1))
)

crop_agg["total_calories_visit"] = (
    crop_agg
    .groupby(key_visit)["calories"]
    .transform(lambda s: s.sum(min_count=1))
)

crop_agg["share_area"] = (
    crop_agg["sum_crop_area"] / crop_agg["total_planted_area_visit"]
)

crop_agg.loc[crop_agg["total_planted_area_visit"].le(0), "share_area"] = pd.NA

crop_agg["share_calories"] = (
    crop_agg["calories"] / crop_agg["total_calories_visit"]
)

crop_agg.loc[crop_agg["total_calories_visit"].le(0), "share_calories"] = pd.NA

In [5]:
# Pivot all crop variables wide

metrics = [
    "dummy_planted",
    "sum_crop_area",
    "sum_harvest_kg",
    "share_area",
    "calories",
    "share_calories",
]

wide_parts = []

presence_wide = (
    crop_agg
    .pivot(index=key_visit, columns="CROP_ID", values="dummy_planted")
    .reindex(columns=crop_keep)
)

for metric in metrics:
    part = (
        crop_agg
        .pivot(index=key_visit, columns="CROP_ID", values=metric)
        .reindex(columns=crop_keep)
    )

    # Absent crop for that HHID-wave-visit -> 0
    part = part.where(presence_wide.notna(), 0)

    if metric == "dummy_planted":
        part = part.fillna(0).astype("Int64")

    part.columns = [
        f"{metric}_{crop_id}"
        for crop_id in part.columns
    ]

    wide_parts.append(part)

crop_panel_wide = pd.concat(wide_parts, axis=1).reset_index()

visit_totals = (
    crop_agg
    .groupby(key_visit, as_index=False)
    .agg(
        total_planted_area_visit=("sum_crop_area", sum_with_missing),
        total_calories_visit=("calories", sum_with_missing),
    )
)

crop_panel_wide = crop_panel_wide.merge(
    visit_totals,
    on=key_visit,
    how="left",
    validate="1:1"
)

print("Crop long shape:", crop_panel_long.shape)
print("Filtered/aggregated crop shape:", crop_agg.shape)
print("Crop wide shape:", crop_panel_wide.shape)

display(crop_panel_wide.head())

Crop long shape: (117920, 19)
Filtered/aggregated crop shape: (115497, 13)
Crop wide shape: (31259, 293)


,Wave,HHID,SEASON,dummy_planted_111,dummy_planted_112,dummy_planted_120,dummy_planted_130,dummy_planted_141,dummy_planted_150,dummy_planted_210,...,share_calories_790,share_calories_810,share_calories_820,share_calories_830,share_calories_840,share_calories_860,share_calories_870,share_calories_890,total_planted_area_visit,total_calories_visit
0,1,1.021E+11,1,0,0,0,1,0,0,1,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,5.25,NaN
1,1,1.021E+11,2,0,0,0,1,0,0,1,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,5.45,NaN
2,1,1.033E+11,1,0,0,0,1,0,0,1,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,6.25,NaN
3,1,1.033E+11,2,0,0,0,1,0,0,1,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,9.25,NaN
4,1,1.043E+11,1,0,0,0,1,0,0,1,...,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,14.15,NaN


In [6]:
# Convert visit to actual cropping season and add YEAR

season_year_map = pd.DataFrame({
    "Wave":  [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 7, 7, 8, 8],
    "VISIT": [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2],
    "CROPPING_SEASON": [1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1],
    "YEAR": [2009, 2009, 2010, 2010, 2011, 2011, 2013, 2013, 2014, 2015, 2017, 2018, 2018, 2019],
})

crop_panel_wide = crop_panel_wide.copy()

# current column is named SEASON but actually stores visit
crop_panel_wide = crop_panel_wide.rename(columns={"SEASON": "VISIT"})

crop_panel_wide["Wave"] = pd.to_numeric(crop_panel_wide["Wave"], errors="coerce").astype("Int64")
crop_panel_wide["VISIT"] = pd.to_numeric(crop_panel_wide["VISIT"], errors="coerce").astype("Int64")

crop_panel_wide = crop_panel_wide.merge(
    season_year_map,
    on=["Wave", "VISIT"],
    how="left",
    validate="m:1"
)

missing_map = crop_panel_wide[
    crop_panel_wide["CROPPING_SEASON"].isna()
][["Wave", "VISIT"]].drop_duplicates()

print("Unmapped Wave-VISIT combinations:", len(missing_map))
display(missing_map)

front_cols = ["Wave", "YEAR", "HHID", "VISIT", "CROPPING_SEASON"]
front_cols = [c for c in front_cols if c in crop_panel_wide.columns]
other_cols = [c for c in crop_panel_wide.columns if c not in front_cols]

crop_panel_wide = crop_panel_wide[front_cols + other_cols]

display(crop_panel_wide.head())

Unmapped Wave-VISIT combinations: 0


,Wave,VISIT


,Wave,YEAR,HHID,VISIT,CROPPING_SEASON,dummy_planted_111,dummy_planted_112,dummy_planted_120,dummy_planted_130,dummy_planted_141,...,share_calories_790,share_calories_810,share_calories_820,share_calories_830,share_calories_840,share_calories_860,share_calories_870,share_calories_890,total_planted_area_visit,total_calories_visit
0,1,2009,1.021E+11,1,1,0,0,0,1,0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,5.25,NaN
1,1,2009,1.021E+11,2,2,0,0,0,1,0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,5.45,NaN
2,1,2009,1.033E+11,1,1,0,0,0,1,0,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,6.25,NaN
3,1,2009,1.033E+11,2,2,0,0,0,1,0,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,9.25,NaN
4,1,2009,1.043E+11,1,1,0,0,0,1,0,...,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,14.15,NaN


In [7]:
# Add hh_id_obs and coordinates to crop_panel_wide


from pathlib import Path
from decimal import Decimal, InvalidOperation
import pandas as pd

HARM_FILE = Path(r"C:/Users/Carl/Desktop/harmonized lsms/largeharmdataset.xlsx")

def clean_merge_id_basic(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() == "nan":
        return pd.NA

    try:
        if "e" in s.lower():
            s = format(Decimal(s), "f")

        if s.endswith(".0"):
            s = s[:-2]

    except InvalidOperation:
        pass

    return s.upper()


def clean_hhid_for_harm_merge(x, wave):
    s = clean_merge_id_basic(x)

    if pd.isna(s):
        return pd.NA

    wave_num = pd.to_numeric(wave, errors="coerce")

    # Wave 4 agriculture HHIDs sometimes appear as 11010401
    # but harmonized data uses H01101-04-01.
    if wave_num == 4:
        if s.startswith("H"):
            return s

        if s.isdigit():
            s = s.zfill(9)
            return f"H{s[:5]}-{s[5:7]}-{s[7:9]}"

    return s

harm = pd.read_excel(
    HARM_FILE,
    sheet_name=0,
    usecols=["wave", "hh_id_merge", "hh_id_obs", "lat_modified", "lon_modified"],
    dtype={
        "wave": str,
        "hh_id_merge": str,
        "hh_id_obs": str,
    }
)

harm["Wave_clean"] = pd.to_numeric(harm["wave"], errors="coerce").astype("Int64")
harm["HHID_clean"] = harm.apply(
    lambda r: clean_hhid_for_harm_merge(r["hh_id_merge"], r["wave"]),
    axis=1
)

harm_map = (
    harm[
        [
            "Wave_clean",
            "HHID_clean",
            "hh_id_obs",
            "lat_modified",
            "lon_modified",
        ]
    ]
    .drop_duplicates()
)

conflicts = (
    harm_map
    .groupby(["Wave_clean", "HHID_clean"], dropna=False)
    .agg(
        n_hh_id_obs=("hh_id_obs", "nunique"),
        n_lat=("lat_modified", "nunique"),
        n_lon=("lon_modified", "nunique"),
    )
    .reset_index()
    .query("n_hh_id_obs > 1 or n_lat > 1 or n_lon > 1")
)

print("Conflicting Wave-HHID mappings:", len(conflicts))
display(conflicts.head(20))

harm_map = (
    harm_map
    .sort_values(["Wave_clean", "HHID_clean"])
    .groupby(["Wave_clean", "HHID_clean"], as_index=False)
    .agg(
        hh_id_obs=("hh_id_obs", lambda s: s.dropna().iloc[0] if len(s.dropna()) else pd.NA),
        lat_modified=("lat_modified", lambda s: s.dropna().iloc[0] if len(s.dropna()) else pd.NA),
        lon_modified=("lon_modified", lambda s: s.dropna().iloc[0] if len(s.dropna()) else pd.NA),
    )
)

crop_panel_wide = crop_panel_wide.copy()

crop_panel_wide["Wave_clean"] = pd.to_numeric(
    crop_panel_wide["Wave"],
    errors="coerce"
).astype("Int64")

crop_panel_wide["HHID_clean"] = crop_panel_wide.apply(
    lambda r: clean_hhid_for_harm_merge(r["HHID"], r["Wave"]),
    axis=1
)

before_rows = len(crop_panel_wide)

crop_panel_wide = crop_panel_wide.merge(
    harm_map,
    on=["Wave_clean", "HHID_clean"],
    how="left",
    validate="m:1",
    indicator="harm_merge_status"
)

after_rows = len(crop_panel_wide)

if before_rows != after_rows:
    raise ValueError(f"Row count changed during merge: {before_rows} -> {after_rows}")

print("Merge status:")
display(crop_panel_wide["harm_merge_status"].value_counts(dropna=False))

crop_panel_wide = crop_panel_wide.drop(
    columns=["Wave_clean", "HHID_clean", "harm_merge_status"]
)

# Move new columns after HHID
new_cols = ["hh_id_obs", "lat_modified", "lon_modified"]

ordered_cols = []
for col in crop_panel_wide.columns:
    ordered_cols.append(col)

    if col == "HHID":
        ordered_cols.extend([c for c in new_cols if c in crop_panel_wide.columns])

ordered_cols = list(dict.fromkeys(ordered_cols))
remaining = [c for c in crop_panel_wide.columns if c not in ordered_cols]

crop_panel_wide = crop_panel_wide[ordered_cols + remaining]

display(crop_panel_wide.head())

Conflicting Wave-HHID mappings: 1


,Wave_clean,HHID_clean,n_hh_id_obs,n_lat,n_lon
11660,5,H0010101,1,2,1


Merge status:


harm_merge_status
both          30506
left_only       753
right_only        0
Name: count, dtype: int64

,Wave,YEAR,HHID,hh_id_obs,lat_modified,lon_modified,VISIT,CROPPING_SEASON,dummy_planted_111,dummy_planted_112,...,share_calories_790,share_calories_810,share_calories_820,share_calories_830,share_calories_840,share_calories_860,share_calories_870,share_calories_890,total_planted_area_visit,total_calories_visit
0,1,2009,1.021E+11,NaN,NaN,NaN,1,1,0,0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,5.25,NaN
1,1,2009,1.021E+11,NaN,NaN,NaN,2,2,0,0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,5.45,NaN
2,1,2009,1.033E+11,NaN,NaN,NaN,1,1,0,0,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,6.25,NaN
3,1,2009,1.033E+11,NaN,NaN,NaN,2,2,0,0,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,9.25,NaN
4,1,2009,1.043E+11,NaN,NaN,NaN,1,1,0,0,...,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,14.15,NaN


In [8]:
OUT = ROOT / "Built panels"
OUT.mkdir(exist_ok=True)

crop_panel_wide.to_excel(
    OUT / "AG_crop_panel_wide_hh_id_obs.xlsx",
    index=False
)

crop_panel_wide.to_csv(
    OUT / "AG_crop_panel_wide_hh_id_obs.csv",
    index=False
)

crop_panel_wide.to_parquet(
    OUT / "AG_crop_panel_wide_hh_id_obs.parquet",
    index=False
)

In [6]:
crop_frequency_table.to_excel(
    ROOT / "Built panels" / "AG_crop_code_frequency_table.xlsx",
    index=False
)

In [3]:
ag5_harvest_check = ag5.copy()

ag5_harvest_check["A5AQ6A_num"] = pd.to_numeric(ag5_harvest_check["A5AQ6A"], errors="coerce")
ag5_harvest_check["A5AQ6D_num"] = pd.to_numeric(ag5_harvest_check["A5AQ6D"], errors="coerce")
ag5_harvest_check["harvest_kg"] = (
    ag5_harvest_check["A5AQ6A_num"]
    * ag5_harvest_check["A5AQ6D_num"]
)

print("Raw row-level harvest kg summary:")
display(
    ag5_harvest_check["harvest_kg"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999])
)

print("Top 50 raw harvest kg rows:")
display(
    ag5_harvest_check
    .sort_values("harvest_kg", ascending=False)
    [[
        "Wave", "SEASON", "HHID", "PARCEL_ID", "PLOT_ID", "CROP_ID",
        "A5AQ6A", "A5AQ6C", "A5AQ6D", "harvest_kg",
        "SOURCE_FILE", "SOURCE_SECTION"
    ]]
    .head(50)
)

Raw row-level harvest kg summary:


count    1.143200e+05
mean     1.148911e+03
std      1.121296e+05
min      0.000000e+00
50%      1.200000e+02
75%      3.000000e+02
90%      7.700000e+02
95%      1.300000e+03
99%      4.000000e+03
99.5%    6.770250e+03
99.9%    3.940928e+04
max      2.510000e+07
Name: harvest_kg, dtype: float64

Top 50 raw harvest kg rows:


,Wave,SEASON,HHID,PARCEL_ID,PLOT_ID,CROP_ID,A5AQ6A,A5AQ6C,A5AQ6D,harvest_kg,SOURCE_FILE,SOURCE_SECTION
72903,3,2,4033000707.0,1.0,1.0,510,5020.0,85.0,5000.0,25100000.0,AGSEC5B.dta,AGSEC5B
121151,7,1,7f851c2ccbf243319aa0de4ffe7c8bff,1,1,150,147.0,45.0,147000.0,21609000.0,AGSEC5A.dta,AGSEC5A
131610,7,2,fbc0a1a072c84998b1c1fb073965059c,1,3,340,67.0,45.0,260000.0,17420000.0,AGSEC5B.dta,AGSEC5B
1748,1,1,1083000903,2.0,1.0,510,1000.0,85.0,5000.0,5000000.0,AGSEC5A.dta,AGSEC5A
100076,5,1,H0920601,2,3,130,2.0,10.0,860000.0,1720000.0,AGSEC5A.dta,AGSEC5A
123984,7,1,e8e2ecd0dc114864a91e5e8a13246465,2,0,130,1200.0,45.0,990.0,1188000.0,AGSEC5A.dta,AGSEC5A
124541,7,1,fbf8f05d30f94fd095971e1f3eb08cfb,1,1,630,1000.0,40.0,1000.0,1000000.0,AGSEC5A.dta,AGSEC5A
124548,7,1,fbf8f05d30f94fd095971e1f3eb08cfb,2,1,120,1000.0,45.0,1000.0,1000000.0,AGSEC5A.dta,AGSEC5A
122643,7,1,b775483de2874fdb9ec10aabf8f35e4c,2,1,210,435.0,45.0,2017.0,877395.0,AGSEC5A.dta,AGSEC5A
146956,8,2,bbe988d76e74439997133682935ed301,2,1,120,906.0,1.0,906.0,820836.0,AGSEC5B.dta,AGSEC5B


In [8]:
OUT = ROOT / "Built panels"
OUT.mkdir(exist_ok=True)

crop_panel_long_export = crop_panel_long.copy()


front_cols = [
    "Wave",
    "HHID",
    "SEASON",
    "CROP_ID",
    "planted_yes",
    "crop_area",
    "qty_harvested_units",
    "harvest_kg",
    "seed_saved_units",
    "lost_wasted_units",
    "no_harvest_reason_A5Q24",
    "total_area",
    "share_crop_area_of_total_area",
    "total_planted_area_season",
    "share_crop_area_of_planted_area",
    "total_harvest_kg_season",
    "share_crop_harvest_kg",
]

front_cols = [c for c in front_cols if c in crop_panel_long_export.columns]
other_cols = [c for c in crop_panel_long_export.columns if c not in front_cols]

crop_panel_long_export = crop_panel_long_export[front_cols + other_cols]

xlsx_path = OUT / "AG_crop_panel_long_fixed.xlsx"
csv_path = OUT / "AG_crop_panel_long_fixed.csv"
parquet_path = OUT / "AG_crop_panel_long_fixed.parquet"

crop_panel_long_export.to_excel(xlsx_path, index=False)
crop_panel_long_export.to_csv(csv_path, index=False)
crop_panel_long_export.to_parquet(parquet_path, index=False)

print("Exported:")
print("Excel:", xlsx_path)
print("CSV:", csv_path)
print("Parquet:", parquet_path)
print("Shape:", crop_panel_long_export.shape)

Exported:
Excel: C:\Users\Carl\Desktop\CSB_project\Built panels\AG_crop_panel_long_fixed.xlsx
CSV: C:\Users\Carl\Desktop\CSB_project\Built panels\AG_crop_panel_long_fixed.csv
Parquet: C:\Users\Carl\Desktop\CSB_project\Built panels\AG_crop_panel_long_fixed.parquet
Shape: (117920, 19)


In [8]:
# aggregated sec3 variables (fertilizer/pesticides and hh agricultural labor)

ag3 = load_ag("AGSEC3_standardized.csv")

key_ag3 = ["Wave", "HHID", "SEASON"]

def is_one(x):
    if pd.isna(x):
        return False

    s = str(x).strip()
    return s in ["1", "1.0"]


def clean_numeric_value(s, missing_codes=[99999, 99998, 99997, 99996]):
    x = pd.to_numeric(s, errors="coerce")
    return x.mask(x.isin(missing_codes))


def sum_with_missing(s):
    return s.sum(min_count=1)


ag3["PARCEL_ID"] = clean_code(ag3["PARCEL_ID"])
ag3["PLOT_ID"] = clean_code(ag3["PLOT_ID"])

for q in ["A3AQ4", "A3AQ14", "A3AQ26"]:
    q_num = clean_numeric_value(ag3[q])
    ag3[f"{q}_use"] = pd.NA
    ag3.loc[q_num.notna(), f"{q}_use"] = q_num.eq(1).astype(int)

ag3["A3AQ38_num"] = clean_numeric_value(ag3["A3AQ38"])
ag3["A3AQ39_num"] = clean_numeric_value(ag3["A3AQ39"])
ag3["A3AQ43_num"] = clean_numeric_value(ag3["A3AQ43"])

ag3["A3AQ41_labor"] = ag3["A3AQ41"].map(lambda x: 1 if is_one(x) else 0)

ag3_plot_dupes = ag3[
    ag3.duplicated(
        ["Wave", "HHID", "SEASON", "PARCEL_ID", "PLOT_ID"],
        keep=False
    )
].copy()

print("AGSEC3 duplicate Wave-HHID-SEASON-parcel-plot rows:", len(ag3_plot_dupes))
display(ag3_plot_dupes.head(20))

ag3_hh_visit = (
    ag3
    .groupby(key_ag3, as_index=False)
    .agg(
        A3Q4_average_use=("A3AQ4_use", "mean"),
        A3Q14_average_use=("A3AQ14_use", "mean"),
        A3Q26_average_use=("A3AQ26_use", "mean"),
        A3AQ38_average=("A3AQ38_num", "mean"),
        A3Q39_average=("A3AQ39_num", "mean"),
        A3Q41_any_labor=("A3AQ41_labor", "max"),
        A3Q43_sum=("A3AQ43_num", sum_with_missing),
    )
)

dupes = ag3_hh_visit[
    ag3_hh_visit.duplicated(key_ag3, keep=False)
]

print("AGSEC3 HH-visit panel shape:", ag3_hh_visit.shape)
print("Duplicate Wave-HHID-SEASON rows:", len(dupes))

display(
    ag3_hh_visit
    .groupby(["Wave", "SEASON"])
    .size()
    .reset_index(name="rows")
)

display(ag3_hh_visit.head())

AGSEC3 duplicate Wave-HHID-SEASON-parcel-plot rows: 6


,Wave,SEASON,SOURCE_FILE,SOURCE_SECTION,HHID,PARCEL_ID,PLOT_ID,A3AQ4,A3AQ14,A3AQ26,...,A3AQ39,A3AQ41,A3AQ43,A3AQ4_use,A3AQ14_use,A3AQ26_use,A3AQ38_num,A3AQ39_num,A3AQ43_num,A3AQ41_labor
48397,3,2,AGSEC3B.dta,AGSEC3B,411300090606.0,1,1,2.0,2.0,2.0,...,46.0,2.0,NaN,0,0,0,4.0,46.0,NaN,0
48398,3,2,AGSEC3B.dta,AGSEC3B,411300090606.0,1,1,2.0,2.0,2.0,...,49.0,2.0,NaN,0,0,0,3.0,49.0,NaN,0
48399,3,2,AGSEC3B.dta,AGSEC3B,411300090606.0,1,2,2.0,2.0,1.0,...,45.0,2.0,NaN,0,0,1,4.0,45.0,NaN,0
48400,3,2,AGSEC3B.dta,AGSEC3B,411300090606.0,1,2,2.0,1.0,2.0,...,56.0,2.0,NaN,0,1,0,4.0,56.0,NaN,0
48401,3,2,AGSEC3B.dta,AGSEC3B,411300090606.0,1,3,2.0,2.0,2.0,...,NaN,2.0,NaN,0,0,0,0.0,NaN,NaN,0
48402,3,2,AGSEC3B.dta,AGSEC3B,411300090606.0,1,3,2.0,2.0,2.0,...,15.0,2.0,NaN,0,0,0,3.0,15.0,NaN,0


AGSEC3 HH-visit panel shape: (31231, 10)
Duplicate Wave-HHID-SEASON rows: 0


,Wave,SEASON,rows
0,1,1,2346
1,1,2,2348
2,2,1,2109
3,2,2,1934
4,3,1,2217
5,3,2,2055
6,4,1,2394
7,4,2,2169
8,5,1,2446
9,5,2,2215


,Wave,HHID,SEASON,A3Q4_average_use,A3Q14_average_use,A3Q26_average_use,A3AQ38_average,A3Q39_average,A3Q41_any_labor,A3Q43_sum
0,1,1013000204,1,NaN,NaN,NaN,NaN,NaN,0,NaN
1,1,1013000204,2,0.0,0.0,0.0,1.000000,40.000000,1,4000.0
2,1,1021000108,1,1.0,0.0,0.0,1.000000,28.000000,1,30000.0
3,1,1021000108,2,0.0,0.0,0.0,0.666667,15.333333,1,80000.0
4,1,1021000113,1,1.0,0.0,0.0,2.000000,36.000000,1,50000.0


In [10]:
OUT = ROOT / "Built panels"
OUT.mkdir(exist_ok=True)

ag3_hh_visit.to_excel(xlsx_path, index=False)

xlsx_path = OUT / "AG3_inputs_aggregated.xlsx"

print("Exported:")
print("Excel:", xlsx_path)


Exported:
Excel: C:\Users\Carl\Desktop\CSB_project\Built panels\AG3_inputs_aggregated.xlsx
